# TFT Local Training Pipeline

Local replacement for the Vertex AI training pipeline. Trains a TemporalFusionTransformer on NQ 8-range OHLCV bars and uploads the best checkpoint to GCS.

## 0. Configuration

In [20]:
import os
os.environ.pop('GOOGLE_APPLICATION_CREDENTIALS', None)

In [ ]:
import torch
print(torch.__version__)               # sollte auf +cu128 enden
print(torch.cuda.is_available())       # True
print(torch.cuda.get_device_name(0))   # NVIDIA GeForce RTX 5080

In [2]:
# ===== CONFIGURE THESE =====
from pathlib import Path

SYMBOL = "GC"  # Instrument represented by the training data
DATA_CSV_RELATIVE_PATH = (
    Path("data")
    / "Candelexport_GC_M1_25_11_2018_20_10_2026"
    / "E:\Programierung\Daten\Candelexport_GC_M1_25_11_2018_20_10_2026.csv"
)
DATA_CSV_CANDIDATES = [
    root / DATA_CSV_RELATIVE_PATH
    for root in (Path.cwd(), *Path.cwd().parents)
]
DATA_CSV_PATH = next(
    (path.resolve() for path in DATA_CSV_CANDIDATES if path.is_file()),
    None,
)
if DATA_CSV_PATH is None:
    searched_paths = ", ".join(str(path) for path in DATA_CSV_CANDIDATES)
    raise FileNotFoundError(f"Training CSV not found. Searched: {searched_paths}")

# Training hyperparameters
MAX_EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 0.001
HIDDEN_SIZE = 64
ATTENTION_HEAD_SIZE = 4
DROPOUT = 0.1
HIDDEN_CONTINUOUS_SIZE = 32
PATIENCE = 10               # Early stopping patience

# Model architecture
MAX_ENCODER_LENGTH = 60  # Input range bars
TARGET_HORIZON_BARS = 60
MAX_PREDICTION_LENGTH = TARGET_HORIZON_BARS

# GCS upload settings
GCS_BUCKET = "tft-for-trading-brains"
GCS_MODEL_PATH = f"tft_checkpoint_{SYMBOL}_latest.ckpt"

# Use GPU if available
import torch
ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
print(f"Training on: {ACCELERATOR}")
print(f"Symbol: {SYMBOL}, Epochs: {MAX_EPOCHS}")
print(f"DATA_CSV_PATH: {DATA_CSV_PATH}")

Training on: cpu
Symbol: GC, Epochs: 20
DATA_CSV_PATH: E:\Programierung\Daten\Candelexport_GC_M1_25_11_2018_20_10_2026.csv


## 1. Install / Import Dependencies

In [3]:
#%pip install google-cloud-storage google-cloud-bigquery

In [4]:
%pip install --upgrade ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import gc
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, GroupNormalizer, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss, MultiHorizonMetric
import sys
import subprocess
from google.cloud import storage, bigquery

torch.set_float32_matmul_precision("medium")

print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
pd.set_option("display.max_columns", None)

PyTorch 2.1.0+cpu, CUDA: False


## 2. Custom Loss Function

In [6]:
class DirectionAwareQuantileLoss(MultiHorizonMetric):
    """
    Composite loss: QuantileLoss + direction penalty.
    Penalizes predictions where the predicted direction of change
    disagrees with the actual direction of change.
    """

    def __init__(
        self,
        quantile_weight: float = 0.7,
        direction_weight: float = 0.3,
        quantiles: list = [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98],
        **kwargs,
    ):
        super().__init__(quantiles=quantiles, **kwargs)
        self.quantile_weight = quantile_weight
        self.direction_weight = direction_weight
        self.quantiles = quantiles

    def loss(self, y_pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        losses = []
        for i, q in enumerate(self.quantiles):
            errors = target - y_pred[..., i]
            q_loss = torch.max((q - 1) * errors, q * errors)
            losses.append(q_loss.unsqueeze(-1))
        quantile_loss = torch.cat(losses, dim=-1).mean(dim=-1)

        median_idx = len(self.quantiles) // 2
        pred_median = y_pred[..., median_idx]

        pred_change = pred_median[:, 1:] - pred_median[:, :-1]
        actual_change = target[:, 1:] - target[:, :-1]

        direction_agreement = torch.tanh(pred_change * 10) * torch.tanh(actual_change * 10)
        direction_penalty = torch.clamp(1.0 - direction_agreement, min=0.0) / 2.0

        pad = torch.zeros_like(direction_penalty[:, :1])
        direction_penalty = torch.cat([pad, direction_penalty], dim=1)

        return self.quantile_weight * quantile_loss + self.direction_weight * direction_penalty

### Data Load from the NQ 8-Range CSV

Each row is one NQ 8-range bar. The CSV is loaded once and reused for both training tasks.

In [7]:
def load_nq_range_csv(csv_path: str | Path = DATA_CSV_PATH) -> pd.DataFrame:
    """Load all NQ 8-range CSV columns in source Bar order."""
    raw = pd.read_csv(
        csv_path,
        sep=";",
        dtype={
            "Bar": "int64",
            "Open": "float32",
            "High": "float32",
            "Low": "float32",
            "Close": "float32",
            "Volume": "float32",
        },
        encoding="utf-8-sig",
    )
    if raw.empty:
        raise ValueError(f"No rows found in NQ CSV: {csv_path}")

    required_values = ["Time UTC", "Open", "High", "Low", "Close", "Volume"]
    if raw[required_values].isna().any().any():
        raise ValueError(f"NQ CSV contains missing timestamps or OHLCV values: {csv_path}")

    # Preserve the source sequence: timestamps can occasionally reverse by a few seconds.
    if not raw["Bar"].is_monotonic_increasing:
        raw = raw.sort_values("Bar", kind="stable")

    df = raw.rename(columns={
        "Time UTC": "timestamp",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume",
    })
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], format="%d.%m.%Y %H:%M:%S", errors="raise", utc=True
    )
    df = df.reset_index(drop=True)
    print(f"Loaded {len(df):,} NQ 8-range bars from {df['timestamp'].min()} to {df['timestamp'].max()}")
    return df


df_raw = load_nq_range_csv()
LOOKBACK_DAYS = (df_raw["timestamp"].max() - df_raw["timestamp"].min()).days
df_raw = df_raw.drop(columns=["Time local"])
df_raw.head()

Loaded 2,796,096 NQ 8-range bars from 2018-11-25 23:00:00+00:00 to 2026-10-20 20:59:00+00:00


,Bar,timestamp,open,close,high,low,Ask,Bid,Delta,Delta/Volume,Session Delta,Session Delta %,Max Delta,Min Delta,Delta Change,volume,Volume/sec,Session Volume,Trades,Height,Duration,Candel MaxVolumePrice,Candel MaxVolumePrice Bid,Candel MaxVolumePrice ASK,Candel VAH,Candel VAL
0,0,2018-11-25 23:00:00+00:00,1223.900024,1224.400024,1224.400024,1223.699951,187,48,139,59.15%,139,59.15%,139,87,0,235.0,0.000000,235,128,0.7,59,1223.9,17,114,1224.2,1223.9
1,1,2018-11-25 23:01:00+00:00,1224.300049,1224.300049,1224.500000,1224.199951,49,54,-5,-4.85%,134,39.64%,24,-8,-144,103.0,1.861255,338,76,0.3,55,1224.3,34,12,1224.4,1224.3
2,2,2018-11-25 23:02:00+00:00,1224.300049,1224.199951,1224.400024,1224.099976,39,66,-27,-25.71%,107,24.15%,-1,-58,-22,105.0,1.765566,443,85,0.3,59,1224.2,30,13,1224.3,1224.2
3,3,2018-11-25 23:03:00+00:00,1224.300049,1224.000000,1224.300049,1224.000000,21,60,-39,-48.15%,68,12.98%,7,-46,-12,81.0,1.446842,524,58,0.3,55,1224.1,33,1,1224.2,1224.0
4,4,2018-11-25 23:04:00+00:00,1223.900024,1224.099976,1224.099976,1223.800049,37,20,17,29.82%,85,14.63%,17,-4,56,57.0,1.341240,581,50,0.3,42,1223.9,19,20,1224.0,1223.9


In [8]:
df_raw.shape

(2796096, 26)

In [9]:
df_raw.columns

Index(['Bar', 'timestamp', 'open', 'close', 'high', 'low', 'Ask', 'Bid',
       'Delta', 'Delta/Volume', 'Session Delta', 'Session Delta %',
       'Max Delta', 'Min Delta', 'Delta Change', 'volume', 'Volume/sec',
       'Session Volume', 'Trades', 'Height', 'Duration',
       'Candel MaxVolumePrice', 'Candel MaxVolumePrice Bid',
       'Candel MaxVolumePrice ASK', 'Candel VAH', 'Candel VAL'],
      dtype='object')

## 3. Loaded NQ 8-Range Bars

In [10]:
print(f"CSV source: {DATA_CSV_PATH}")
print(f"Bars loaded: {len(df_raw):,}")
print(f"Date range (UTC): {df_raw['timestamp'].min()} to {df_raw['timestamp'].max()}")

CSV source: E:\Programierung\Daten\Candelexport_GC_M1_25_11_2018_20_10_2026.csv
Bars loaded: 2,796,096
Date range (UTC): 2018-11-25 23:00:00+00:00 to 2026-10-20 20:59:00+00:00


In [11]:
df_raw.tail()

,Bar,timestamp,open,close,high,low,Ask,Bid,Delta,Delta/Volume,Session Delta,Session Delta %,Max Delta,Min Delta,Delta Change,volume,Volume/sec,Session Volume,Trades,Height,Duration,Candel MaxVolumePrice,Candel MaxVolumePrice Bid,Candel MaxVolumePrice ASK,Candel VAH,Candel VAL
2796091,2796091,2026-10-20 20:55:00+00:00,4142.600098,4143.200195,4143.399902,4142.500000,27,23,4,8.00%,32,0.02%,12,-2,-1,50.0,0.929824,141664,47,0.9,53,4142.9,5,8,4143.2,4142.6
2796092,2796092,2026-10-20 20:56:00+00:00,4143.000000,4142.799805,4143.399902,4142.700195,28,34,-6,-9.68%,26,0.02%,6,-8,-10,62.0,1.076471,141726,48,0.7,57,4143.2,14,7,4143.3,4142.9
2796093,2796093,2026-10-20 20:57:00+00:00,4142.799805,4143.399902,4143.399902,4142.799805,18,21,-3,-7.69%,23,0.02%,4,-6,3,39.0,0.654458,141765,36,0.6,59,4143.3,5,6,4143.3,4143.0
2796094,2796094,2026-10-20 20:58:00+00:00,4143.200195,4143.700195,4143.799805,4143.200195,22,14,8,22.22%,31,0.02%,9,-2,11,36.0,0.608097,141801,36,0.6,59,4143.5,2,6,4143.7,4143.2
2796095,2796095,2026-10-20 20:59:00+00:00,4143.700195,4142.500000,4143.700195,4142.500000,29,42,-13,-18.31%,18,0.01%,16,-13,-21,71.0,1.193693,141872,66,1.2,59,4143.5,5,8,4143.6,4142.6


## 4. Feature Engineering

In [12]:
def calculate_all_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate all 76 features required for the TFT model.
    Input df must have: timestamp, open, high, low, close, volume.
    Lag windows count NQ range bars; legacy minute-suffixed feature names are retained for compatibility.
    """
    df = df.copy()

    # 1. RETURNS (5 features)
    df['returns_1m'] = df['close'].pct_change(1)
    df['returns_5m'] = df['close'].pct_change(5)
    df['returns_15m'] = df['close'].pct_change(15)
    df['returns_30m'] = df['close'].pct_change(30)
    df['returns_60m'] = df['close'].pct_change(60)

    # 2. PRICE RATIOS (6 features)
    df['high_low_ratio'] = df['high'] / df['low']
    df['close_open_ratio'] = df['close'] / df['open']
    df['high_close_ratio'] = df['high'] / df['close']
    df['low_close_ratio'] = df['low'] / df['close']
    df['upper_shadow'] = (df['high'] - np.maximum(df['open'], df['close'])) / (df['high'] - df['low'] + 1e-10)
    df['lower_shadow'] = (np.minimum(df['open'], df['close']) - df['low']) / (df['high'] - df['low'] + 1e-10)

    # 3. SMA FEATURES (11 features)
    for period in [5, 10, 20, 30]:
        sma = df['close'].rolling(window=period).mean()
        df[f'sma_{period}_slope'] = sma.pct_change()
        df[f'close_to_sma_{period}'] = (df['close'] - sma) / sma

    sma_60 = df['close'].rolling(window=60).mean()
    df['close_to_sma_60'] = (df['close'] - sma_60) / sma_60

    sma_120 = df['close'].rolling(window=120).mean()
    df['sma_120_slope'] = sma_120.pct_change()
    df['close_to_sma_120'] = (df['close'] - sma_120) / sma_120

    # 4. MACD (2 features)
    ema_12 = df['close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd'] = ema_12 - ema_26
    signal_line = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_histogram'] = df['macd'] - signal_line

    # 5. VOLATILITY (4 features)
    df['volatility_5'] = df['returns_1m'].rolling(window=5).std()
    df['volatility_10'] = df['returns_1m'].rolling(window=10).std()
    df['volatility_20'] = df['returns_1m'].rolling(window=20).std()
    df['volatility_60'] = df['returns_1m'].rolling(window=60).std()

    # 6. ATR (2 features)
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    tr = np.maximum(high_low, np.maximum(high_close, low_close))
    df['atr_14'] = tr.rolling(window=14).mean()
    df['atr_60'] = tr.rolling(window=60).mean()

    # 7. BOLLINGER BANDS (4 features)
    for period in [20, 60]:
        sma = df['close'].rolling(window=period).mean()
        std = df['close'].rolling(window=period).std()
        df[f'bb_std_{period}'] = std
        df[f'bb_position_{period}'] = (df['close'] - sma) / (2 * std + 1e-10)

    # 8. RSI (3 features)
    def calc_rsi(series, period):
        delta = series.diff()
        gain = delta.where(delta > 0, 0).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / (loss + 1e-10)
        return 100 - (100 / (1 + rs))

    df['rsi_14'] = calc_rsi(df['close'], 14)
    df['rsi_20'] = calc_rsi(df['close'], 20)
    df['rsi_60'] = calc_rsi(df['close'], 60)

    # 9. STOCHASTIC (3 features)
    for period in [14, 60]:
        lowest_low = df['low'].rolling(window=period).min()
        highest_high = df['high'].rolling(window=period).max()
        df[f'stoch_k_{period}'] = 100 * (df['close'] - lowest_low) / (highest_high - lowest_low + 1e-10)
    df['stoch_d_14'] = df['stoch_k_14'].rolling(window=3).mean()

    # 10. RATE OF CHANGE (2 features)
    df['roc_10'] = df['close'].pct_change(10) * 100
    df['roc_20'] = df['close'].pct_change(20) * 100

    # 11. VOLUME INDICATORS (12 features)
    df['volume_change'] = df['volume'].pct_change(1)
    df['volume_change_5m'] = df['volume'].pct_change(5)

    for period in [5, 10, 20, 60]:
        df[f'volume_sma_{period}'] = df['volume'].rolling(window=period).mean()
        df[f'volume_ratio_{period}'] = df['volume'] / (df[f'volume_sma_{period}'] + 1e-10)

    # 12. OBV, VPT, MFI (5 features)
    obv = np.where(df['close'] > df['close'].shift(), df['volume'],
                   np.where(df['close'] < df['close'].shift(), -df['volume'], 0))
    df['obv'] = np.cumsum(obv)
    obv_sma = pd.Series(df['obv']).rolling(window=20).mean()
    df['obv_ratio'] = df['obv'] / (obv_sma + 1e-10)

    df['vpt'] = (df['volume'] * df['close'].pct_change()).cumsum()

    def calc_mfi(df, period):
        typical_price = (df['high'] + df['low'] + df['close']) / 3
        money_flow = typical_price * df['volume']
        positive_flow = money_flow.where(typical_price > typical_price.shift(), 0).rolling(window=period).sum()
        negative_flow = money_flow.where(typical_price < typical_price.shift(), 0).rolling(window=period).sum()
        mfi = 100 - (100 / (1 + positive_flow / (negative_flow + 1e-10)))
        return mfi

    df['mfi_14'] = calc_mfi(df, 14)
    df['mfi_60'] = calc_mfi(df, 60)

    # 13. VWAP RATIOS (2 features)
    vwap_20 = (df['volume'] * df['close']).rolling(window=20).sum() / (df['volume'].rolling(window=20).sum() + 1e-10)
    vwap_60 = (df['volume'] * df['close']).rolling(window=60).sum() / (df['volume'].rolling(window=60).sum() + 1e-10)
    df['close_to_vwap_20'] = (df['close'] - vwap_20) / vwap_20
    df['close_to_vwap_60'] = (df['close'] - vwap_60) / vwap_60

    # 14. TIME FEATURES (10 features)
    df['hour'] = df['timestamp'].dt.hour
    df['minute'] = df['timestamp'].dt.minute
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['is_morning'] = ((df['hour'] >= 9) & (df['hour'] < 12)).astype(int)
    df['is_afternoon'] = ((df['hour'] >= 12) & (df['hour'] < 16)).astype(int)

    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    # 15. VOLUME LAGS (5 features)
    df['volume_lag_1'] = df['volume'].shift(1)
    df['volume_lag_5'] = df['volume'].shift(5)
    df['volume_lag_15'] = df['volume'].shift(15)
    df['volume_lag_30'] = df['volume'].shift(30)
    df['volume_lag_60'] = df['volume'].shift(60)

    # 16. TARGET VARIABLES (2 features)
    # Keep legacy target names; the 60-step horizon is measured in NQ bars, not minutes.
    df['target_close_60m'] = df['close'].shift(-TARGET_HORIZON_BARS)
    df['target_return_60m'] = df['close'].pct_change(TARGET_HORIZON_BARS).shift(-TARGET_HORIZON_BARS)

    return df

In [13]:
def remove_highly_correlated_features(df: pd.DataFrame, threshold: float = 0.95):
    """Remove features with pairwise correlation > threshold."""
    exclude_cols = [
        "timestamp", "time_idx", "group",
        "target_close_60m", "target_return_60m",
        "open", "high", "low", "close", "volume", "vwap",
    ]
    feature_cols = [
        c for c in df.columns
        if c not in exclude_cols and df[c].dtype in ["float64", "float32", "int64"]
    ]

    df_clean = df[feature_cols].dropna()
    corr_matrix = df_clean.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]

    print(f"Correlation filtering (threshold={threshold}):")
    print(f"  Features before: {len(feature_cols)}")
    print(f"  Features dropped: {len(to_drop)} — {to_drop[:10]}{'...' if len(to_drop) > 10 else ''}")
    print(f"  Features remaining: {len(feature_cols) - len(to_drop)}")

    return df.drop(columns=to_drop), to_drop


def prepare_dataset(df: pd.DataFrame):
    """Run feature engineering and prepare the TFT-ready DataFrame."""
    # Calculate VWAP
    if df['volume'].sum() > 0:
        df["vwap"] = (df["volume"] * (df["high"] + df["low"] + df["close"]) / 3).cumsum() / df["volume"].cumsum()
    else:
        df["vwap"] = (df["high"] + df["low"] + df["close"]) / 3

    # Feature engineering
    df_feat = calculate_all_features(df)

    # Drop rows with NaN from indicators
    target_cols = ["target_close_60m", "target_return_60m"]
    non_target = [c for c in df_feat.columns if c not in target_cols]
    df_feat = df_feat.dropna(subset=non_target)
    df_feat = df_feat.dropna(subset=target_cols)
    df_feat = df_feat.reset_index(drop=True)

    # Correlation filtering
    df_feat, dropped_cols = remove_highly_correlated_features(df_feat, threshold=0.95)

    # Add required columns
    df_feat["time_idx"] = range(len(df_feat))
    df_feat["group"] = "default"

    # Clean infinities
    numeric_cols = df_feat.select_dtypes(include=[np.number]).columns.tolist()
    df_feat[numeric_cols] = df_feat[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df_feat[numeric_cols] = df_feat[numeric_cols].ffill().bfill()

    print(f"\nPrepared dataset: {len(df_feat):,} rows, {len(df_feat.columns)} columns")
    return df_feat

In [14]:
# Run feature engineering
df_tft = prepare_dataset(df_raw)

# Save metadata
data_start = str(df_raw['timestamp'].min())
data_end = str(df_raw['timestamp'].max())
print(f"Data range: {data_start} to {data_end}")

# Free raw data
del df_raw
gc.collect()

C:\Users\danie\AppData\Local\Temp\ipykernel_40700\774690813.py:27: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Call ffill before calling pct_change to retain current behavior and silence this warning.
  df[f'sma_{period}_slope'] = sma.pct_change()
C:\Users\danie\AppData\Local\Temp\ipykernel_40700\774690813.py:27: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Call ffill before calling pct_change to retain current behavior and silence this warning.
  df[f'sma_{period}_slope'] = sma.pct_change()
C:\Users\danie\AppData\Local\Temp\ipykernel_40700\774690813.py:27: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Call ffill before calling pct_change to retain current behavior and silence this warning.
  df[f'sma_{period}_slope'] = sma.pct_change()
C:\Users\danie\AppData\Lo

Correlation filtering (threshold=0.95):
  Features before: 88
  Features dropped: 12 — ['Volume/sec', 'Trades', 'Candel VAH', 'Candel VAL', 'close_open_ratio', 'sma_5_slope', 'sma_30_slope', 'close_to_sma_30', 'roc_10', 'roc_20']...
  Features remaining: 76

Prepared dataset: 2,795,916 rows, 94 columns
Data range: 2018-11-25 23:00:00+00:00 to 2026-10-20 20:59:00+00:00


111

In [15]:
df_tft.head()

,Bar,timestamp,open,close,high,low,Ask,Bid,Delta,Delta/Volume,Session Delta,Session Delta %,Max Delta,Min Delta,Delta Change,volume,Session Volume,Height,Duration,Candel MaxVolumePrice,Candel MaxVolumePrice Bid,Candel MaxVolumePrice ASK,vwap,returns_1m,returns_5m,returns_15m,returns_30m,returns_60m,high_low_ratio,high_close_ratio,low_close_ratio,upper_shadow,lower_shadow,close_to_sma_5,sma_10_slope,close_to_sma_10,sma_20_slope,close_to_sma_20,close_to_sma_60,sma_120_slope,close_to_sma_120,macd,macd_histogram,volatility_5,volatility_10,volatility_20,volatility_60,atr_14,atr_60,bb_std_20,bb_position_20,bb_std_60,bb_position_60,rsi_14,rsi_20,rsi_60,stoch_k_14,stoch_k_60,stoch_d_14,volume_change,volume_change_5m,volume_sma_5,volume_ratio_5,volume_sma_10,volume_ratio_10,volume_sma_20,volume_ratio_20,volume_sma_60,volume_ratio_60,obv,obv_ratio,vpt,mfi_14,mfi_60,hour,minute,day_of_week,is_morning,is_afternoon,hour_sin,hour_cos,minute_sin,minute_cos,day_sin,day_cos,volume_lag_1,volume_lag_5,volume_lag_15,volume_lag_30,volume_lag_60,target_close_60m,target_return_60m,time_idx,group
0,120,2018-11-26 01:02:00+00:00,1222.800049,1223.099976,1223.300049,1222.800049,144,99,45,18.52%,31,0.54%,64,-14,160,243.0,5715,0.5,58,1222.9,40,51,1224.155518,0.000245,-0.000490,-0.000817,-0.001062,-0.001388,1.000409,1.000164,0.999755,0.400146,0.000000,-0.000082,-0.000065,-0.000302,-0.000045,-0.000507,-0.000962,-0.000009,-0.000932,-0.305543,-0.071887,0.000212,0.000199,0.000149,0.000111,0.228585,0.180007,0.360700,-0.859483,0.474102,-1.242729,31.816668,29.628792,35.086630,33.333333,18.182323,13.494347,-0.076046,5.567567,156.4,1.553708,105.0,2.314286,65.10,3.732719,51.533333,4.715395,-891.0,1.381931,-0.158366,37.557133,35.424774,1,2,0,0,0,0.258819,0.965926,0.207912,0.978148,0.0,1.0,263.0,37.0,3.0,7.0,25.0,1224.300049,0.000981,0,default
1,121,2018-11-26 01:03:00+00:00,1223.199951,1223.300049,1223.300049,1223.099976,43,7,36,72.00%,67,1.16%,38,1,-9,50.0,5765,0.2,59,1223.2,1,27,1224.147461,0.000164,-0.000163,-0.000490,-0.000980,-0.001225,1.000164,1.000000,0.999836,0.000000,0.499695,0.000114,-0.000049,-0.000090,-0.000033,-0.000311,-0.000779,-0.000007,-0.000762,-0.295688,-0.049626,0.000236,0.000212,0.000156,0.000113,0.235735,0.180007,0.360700,-0.526701,0.485536,-0.981686,39.130666,35.715220,37.287925,50.005086,27.276258,30.162709,-0.794239,-0.397590,149.8,0.333778,108.5,0.460829,67.00,0.746269,51.900000,0.963391,-841.0,1.273856,-0.150189,40.534853,35.914723,1,3,0,0,0,0.258819,0.965926,0.309017,0.951057,0.0,1.0,243.0,83.0,71.0,7.0,28.0,1224.400024,0.000899,1,default
2,122,2018-11-26 01:04:00+00:00,1223.300049,1223.300049,1223.300049,1223.000000,110,102,8,3.77%,75,1.25%,19,-37,-28,212.0,5977,0.3,59,1223.2,71,36,1224.113892,0.000000,-0.000082,-0.000409,-0.000980,-0.001225,1.000245,1.000000,0.999755,0.000000,1.000000,0.000131,-0.000016,-0.000074,-0.000029,-0.000282,-0.000758,-0.000006,-0.000756,-0.284598,-0.030829,0.000234,0.000188,0.000155,0.000113,0.250026,0.183341,0.361973,-0.476506,0.495425,-0.936860,36.364645,37.037707,37.287925,50.005086,27.276258,44.447835,3.240000,1.585366,175.8,1.205916,119.7,1.771094,76.75,2.762215,54.933333,3.859223,-841.0,1.246295,-0.150189,31.278697,32.814520,1,4,0,0,0,0.258819,0.965926,0.406737,0.913545,0.0,1.0,50.0,82.0,9.0,43.0,30.0,1224.400024,0.000899,2,default
3,123,2018-11-26 01:05:00+00:00,1223.199951,1223.000000,1223.199951,1222.900024,12,44,-32,-57.14%,43,0.71%,-1,-40,-40,56.0,6033,0.3,59,1223.0,19,2,1224.103882,-0.000245,-0.000163,-0.000735,-0.001225,-0.001470,1.000245,1.000163,0.999918,0.000000,0.333333,-0.000082,-0.000065,-0.000253,-0.000041,-0.000486,-0.000979,-0.000007,-0.000994,-0.296602,-0.034266,0.000249,0.000176,0.000162,0.000117,0.271458,0.185008,0.379022,-0.784933,0.514358,-1.164885,32.000391,33.333333,35.483363,25.002543,13.638129,41.670905,-0.735849,-0.495495,164.8,0.339806,115.0,0.486957,77.30,0.724450,53.266667,1.051314,-897.0,1.295868,-0.163924,30.000384,33.919185,1,5,0,0,0,0.258819,0.965926,0.500000,0.866025,0.0,1

## 5. Build TimeSeriesDataSet

In [16]:
def build_feature_lists(df: pd.DataFrame):
    """Derive the time-varying known and unknown reals lists."""
    time_varying_known_reals = ["hour_sin", "hour_cos", "minute_sin", "minute_cos", "day_sin", "day_cos"]
    time_varying_known_reals = [c for c in time_varying_known_reals if c in df.columns]

    exclude = [
        "timestamp", "time_idx", "group",
        "target_close_60m", "target_return_60m",
        "hour", "minute", "day_of_week",
        "is_morning", "is_afternoon",
    ] + time_varying_known_reals

    time_varying_unknown_reals = [
        c for c in df.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
    ]

    if "close" in time_varying_unknown_reals:
        time_varying_unknown_reals.remove("close")
    time_varying_unknown_reals = ["close"] + time_varying_unknown_reals

    print(f"Time-varying known reals: {len(time_varying_known_reals)}")
    print(f"Time-varying unknown reals: {len(time_varying_unknown_reals)}")
    return time_varying_known_reals, time_varying_unknown_reals


time_varying_known_reals, time_varying_unknown_reals = build_feature_lists(df_tft)

# Train/validation split (80/20)
training_cutoff = int(len(df_tft) * 0.8)

print(f"\nDataset size: {len(df_tft):,} rows")
print(f"Training cutoff: time_idx={training_cutoff}")
print(f"Training rows: {len(df_tft[df_tft['time_idx'] <= training_cutoff]):,}")
print(f"Validation rows: {len(df_tft[df_tft['time_idx'] > training_cutoff]):,}")

Time-varying known reals: 6
Time-varying unknown reals: 76

Dataset size: 2,795,916 rows
Training cutoff: time_idx=2236732
Training rows: 2,236,733
Validation rows: 559,183


In [17]:
training = TimeSeriesDataSet(
    df_tft[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="close",
    group_ids=["group"],
    min_encoder_length=MAX_ENCODER_LENGTH // 2,
    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    static_categoricals=["group"],
    time_varying_known_categoricals=[],
    time_varying_known_reals=time_varying_known_reals,
    time_varying_unknown_categoricals=[],
    time_varying_unknown_reals=time_varying_unknown_reals,
    target_normalizer=GroupNormalizer(
        groups=["group"],
        transformation="softplus",
    ),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    df_tft[lambda x: x.time_idx > training_cutoff],
    predict=False,
)

training_rows = len(training)
validation_rows = len(validation)
print(f"Training samples: {training_rows:,}, Validation samples: {validation_rows:,}")

# Free DataFrame
del df_tft
gc.collect()

# Create dataloaders
train_dataloader = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)

Training samples: 2,236,674, Validation samples: 559,124


## 6. Build & Train Model

In [18]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=LEARNING_RATE,
    hidden_size=HIDDEN_SIZE,
    attention_head_size=ATTENTION_HEAD_SIZE,
    dropout=DROPOUT,
    hidden_continuous_size=HIDDEN_CONTINUOUS_SIZE,
    output_size=7,  # 7 quantiles
    loss=DirectionAwareQuantileLoss(quantile_weight=0.7, direction_weight=0.3),
    reduce_on_plateau_patience=4,
)
print(f"Model parameters: {tft.size() / 1e3:.1f}k")

Model parameters: 1024.4k


c:\Users\danie\miniconda3\envs\trading_brains\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\danie\miniconda3\envs\trading_brains\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [19]:
# Callbacks
checkpoint_dir = "checkpoints/training"
os.makedirs(checkpoint_dir, exist_ok=True)

early_stop = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=PATIENCE, verbose=True, mode="min")
lr_monitor = LearningRateMonitor(logging_interval="step")
checkpoint_cb = ModelCheckpoint(
    dirpath=checkpoint_dir,
    monitor="val_loss",
    filename=f"tft-{SYMBOL}" + "-{epoch:02d}-{val_loss:.4f}",
    save_top_k=1,
    mode="min",
)

# Trainer
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    callbacks=[early_stop, lr_monitor, checkpoint_cb],
    enable_model_summary=True,
    accelerator=ACCELERATOR,
    gradient_clip_val=0.1,
)

print(f"Starting training on {ACCELERATOR}...")
trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
c:\Users\danie\miniconda3\envs\trading_brains\Lib\site-packages\lightning\pytorch\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Starting training on cpu...


c:\Users\danie\miniconda3\envs\trading_brains\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:881: Checkpoint directory E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\checkpoints\training exists and is not empty.

   | Name                               | Type                            | Params | Mode  | FLOPs
--------------------------------------------------------------------------------------------------------
0  | loss                               | DirectionAwareQuantileLoss      | 0      | train | 0    
1  | logging_metrics                    | ModuleList                      | 0      | train | 0    
2  | input_embeddings                   | MultiEmbedding                  | 1      | train | 0    
3  | prescalers                         | ModuleDict                      | 5.5 K  | train | 0    
4  | static_variable_selection          | VariableSelectionNetwork        | 20.8 K | train | 0    
5  | encoder_variable_selection       

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\danie\miniconda3\envs\trading_brains\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
c:\Users\danie\miniconda3\envs\trading_brains\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

c:\Users\danie\miniconda3\envs\trading_brains\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
best_model_path = checkpoint_cb.best_model_path
best_val_loss = float(checkpoint_cb.best_model_score)
print(f"Best model: {best_model_path}")
print(f"Best val_loss: {best_val_loss:.4f}")
print(f"Epochs trained: {trainer.current_epoch + 1}")

## 7. Evaluate on Validation Set

In [ ]:
# Load best checkpoint
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path, strict=False)

# Get predictions on validation set
predictions = best_tft.predict(val_dataloader, return_x=True)

if predictions.output.dim() == 3:
    pred_values = predictions.output[:, :, 3]  # median quantile
else:
    pred_values = predictions.output

actuals = predictions.x["decoder_target"]
if actuals.dim() == 3:
    actuals = actuals[:, :, 0]

pred_np = pred_values.detach().cpu().numpy().flatten()
actual_np = actuals.detach().cpu().numpy().flatten()

# Metrics
errors = actual_np - pred_np
mae = float(np.mean(np.abs(errors)))
rmse = float(np.sqrt(np.mean(errors ** 2)))
mask = actual_np != 0
mape = float(np.mean(np.abs(errors[mask] / actual_np[mask])) * 100) if mask.any() else None
ss_res = np.sum(errors ** 2)
ss_tot = np.sum((actual_np - np.mean(actual_np)) ** 2)
r_squared = float(1 - ss_res / ss_tot) if ss_tot != 0 else None

print(f"Validation Metrics:")
print(f"  MAE:   {mae:.4f}")
print(f"  RMSE:  {rmse:.4f}")
print(f"  MAPE:  {mape:.4f}%" if mape else "  MAPE:  N/A")
print(f"  R²:    {r_squared:.4f}" if r_squared else "  R²:    N/A")

## 8. Upload to GCS

In [ ]:
print(f"Uploading checkpoint to gs://{GCS_BUCKET}/{GCS_MODEL_PATH}...")
storage_client = storage.Client()
bucket = storage_client.bucket(GCS_BUCKET)

# Upload as latest
blob = bucket.blob(GCS_MODEL_PATH)
blob.upload_from_filename(best_model_path)

# Upload timestamped copy
timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
timestamped_path = f"tft_checkpoint_{SYMBOL}_{timestamp}.ckpt"
blob_ts = bucket.blob(timestamped_path)
blob_ts.upload_from_filename(best_model_path)

print(f"✓ Uploaded: gs://{GCS_BUCKET}/{GCS_MODEL_PATH}")
print(f"✓ Uploaded: gs://{GCS_BUCKET}/{timestamped_path}")

## 9. Log Metrics to BigQuery

In [ ]:
bq_client = bigquery.Client(project="trading-brains")
table_id = "trading-brains.tft_predictions.retraining_eval_logs"

row = {
    "run_timestamp": datetime.utcnow().isoformat(),
    "symbol": SYMBOL,
    "best_val_loss": round(best_val_loss, 6),
    "mae": round(mae, 6),
    "rmse": round(rmse, 6),
    "mape": round(mape, 6) if mape else None,
    "r_squared": round(r_squared, 6) if r_squared else None,
    "epochs_trained": trainer.current_epoch + 1,
    "max_epochs": MAX_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "hidden_size": HIDDEN_SIZE,
    "attention_head_size": ATTENTION_HEAD_SIZE,
    "dropout": DROPOUT,
    "patience": PATIENCE,
    "lookback_days": LOOKBACK_DAYS,
    "data_start_date": data_start,
    "data_end_date": data_end,
    "training_rows": training_rows,
    "validation_rows": validation_rows,
    "model_gcs_path": f"gs://{GCS_BUCKET}/{GCS_MODEL_PATH}",
}

errors_bq = bq_client.insert_rows_json(table_id, [row])
if errors_bq:
    print(f"BigQuery insert errors: {errors_bq}")
else:
    print(f"✓ Metrics logged to {table_id}")

## 10. Summary

In [ ]:
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"  Symbol:         {SYMBOL}")
print(f"  Epochs:         {trainer.current_epoch + 1}")
print(f"  Best val_loss:  {best_val_loss:.4f}")
print(f"  MAE:            {mae:.4f}")
print(f"  R²:             {r_squared:.4f}" if r_squared else "  R²:  N/A")
print(f"  Data range:     {data_start} → {data_end}")
print(f"  Model:          gs://{GCS_BUCKET}/{GCS_MODEL_PATH}")
print("=" * 60)